In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col, count
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col, sum as _sum, when
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler
import pandas as pd
rom pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator



In [ ]:
import kagglehub

path = kagglehub.dataset_download("jainilcoder/online-payment-fraud-detection")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/online-payment-fraud-detection


In [ ]:
import os

dataset_path = r"/root/.cache/kagglehub/datasets/jainilcoder/online-payment-fraud-detection/versions/1"
print(os.listdir(dataset_path))

['onlinefraud.csv']


In [ ]:

from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Online Fraud Detection") \
    .master("local[*]") \
    .getOrCreate()



df = spark.read.csv(
    r"/root/.cache/kagglehub/datasets/jainilcoder/online-payment-fraud-detection/versions/1/onlinefraud.csv",
    header=True,
    inferSchema=True
)



df.show(5)


+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|   1| PAYMENT| 9839.64|C1231006815|     170136.0|     160296.36|M1979787155|           0.0|           0.0|      0|             0|
|   1| PAYMENT| 1864.28|C1666544295|      21249.0|      19384.72|M2044282225|           0.0|           0.0|      0|             0|
|   1|TRANSFER|   181.0|C1305486145|        181.0|           0.0| C553264065|           0.0|           0.0|      1|             0|
|   1|CASH_OUT|   181.0| C840083671|        181.0|           0.0|  C38997010|       21182.0|           0.0|      1|             0|
|   1| PAYMENT|11668.14|C2048537720|      41554.0|      29885.86|M1230701703|      

In [ ]:
df.printSchema()

# Count non-null values per column

df.select([count(col(c)).alias(c) for c in df.columns]).show()

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)

+-------+-------+-------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   step|   type| amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+-------+-------+-------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|6362620|6362620|6362620| 6362620|      6362620|       6362620| 6362620|       6362620|       6362620|6362620|       6362620|
+-------+-------+-------+----

In [ ]:


# Count nulls in each column
df.select([
    _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|step|type|amount|nameOrig|oldbalanceOrg|newbalanceOrig|nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+
|   0|   0|     0|       0|            0|             0|       0|             0|             0|      0|             0|
+----+----+------+--------+-------------+--------------+--------+--------------+--------------+-------+--------------+



In [ ]:
original_count = df.count()
df_no_dupes = df.dropDuplicates()
new_count = df_no_dupes.count()

print("Any duplicates:", original_count != new_count)

Any duplicates: False


In [ ]:
df = df.drop('nameOrig', 'nameDest', 'isFlaggedFraud', 'step')

In [ ]:
df.head(5)

[Row(type='PAYMENT', amount=9839.64, oldbalanceOrg=170136.0, newbalanceOrig=160296.36, oldbalanceDest=0.0, newbalanceDest=0.0, isFraud=0),
 Row(type='PAYMENT', amount=1864.28, oldbalanceOrg=21249.0, newbalanceOrig=19384.72, oldbalanceDest=0.0, newbalanceDest=0.0, isFraud=0),
 Row(type='TRANSFER', amount=181.0, oldbalanceOrg=181.0, newbalanceOrig=0.0, oldbalanceDest=0.0, newbalanceDest=0.0, isFraud=1),
 Row(type='CASH_OUT', amount=181.0, oldbalanceOrg=181.0, newbalanceOrig=0.0, oldbalanceDest=21182.0, newbalanceDest=0.0, isFraud=1),
 Row(type='PAYMENT', amount=11668.14, oldbalanceOrg=41554.0, newbalanceOrig=29885.86, oldbalanceDest=0.0, newbalanceDest=0.0, isFraud=0)]

In [ ]:
df.select("type").distinct().show()


+--------+
|    type|
+--------+
|TRANSFER|
| CASH_IN|
|CASH_OUT|
| PAYMENT|
|   DEBIT|
+--------+



In [ ]:


indexer = StringIndexer(inputCol="type", outputCol="type_index")
df = indexer.fit(df).transform(df)


In [ ]:
df = df.drop("type").withColumnRenamed("type_index", "type")


In [ ]:


# Assemble all the features into a vector
feature_columns = [col for col in df.columns if col != 'isFraud']  # Exclude 'isFraud' from features
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
df_vector = assembler.transform(df)

# Compute the correlation matrix
correlation_matrix = Correlation.corr(df_vector, 'features').head()[0]

# Convert the correlation matrix to a Pandas DataFrame for sorting
correlation_matrix = pd.DataFrame(correlation_matrix.toArray(), columns=feature_columns, index=feature_columns)

# Get correlations with 'isFraud' (it's not in the matrix, so we calculate it separately)
correlations_with_isFraud = {}
for col_name in feature_columns:
    # Calculate correlation between each feature and 'isFraud'
    correlation = df.stat.corr(col_name, "isFraud")
    correlations_with_isFraud[col_name] = correlation

# Create a Pandas Series for sorting
correlations_with_isFraud_series = pd.Series(correlations_with_isFraud)

# Sort correlations by 'isFraud'
sorted_correlations = correlations_with_isFraud_series.sort_values(ascending=False)
print(sorted_correlations)

amount            0.076688
type              0.016171
oldbalanceOrg     0.010154
newbalanceDest    0.000535
oldbalanceDest   -0.005885
newbalanceOrig   -0.008148
dtype: float64


In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.sql import functions as F

# Encode 'isFraud' column using StringIndexer (equivalent to map function in Pandas)
indexer = StringIndexer(inputCol="isFraud", outputCol="isFraudIndex")
df = indexer.fit(df).transform(df)

# Select feature columns and assemble them into a single vector column
feature_columns = ['type', 'amount', 'oldbalanceOrg', 'newbalanceOrig']
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
df = assembler.transform(df)

# Split the data into training and testing sets (80% train, 20% test)
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# Prepare the input features (X) and target (y) for model training
x_train = train_data.select("features").rdd.map(lambda row: row[0]).collect()
x_test = test_data.select("features").rdd.map(lambda row: row[0]).collect()

y_train = train_data.select("isFraudIndex").rdd.map(lambda row: row[0]).collect()
y_test = test_data.select("isFraudIndex").rdd.map(lambda row: row[0]).collect()

print(f"x_train shape: {len(x_train)}")
print(f"x_test shape: {len(x_test)}")
print(f"y_train shape: {len(y_train)}")
print(f"y_test shape: {len(y_test)}")


x_train shape: 5089484
x_test shape: 1273136
y_train shape: 5089484
y_test shape: 1273136


In [ ]:

rf = RandomForestClassifier(featuresCol='features', labelCol='isFraudIndex', numTrees=10)


rf_model = rf.fit(train_data)


predictions = rf_model.transform(test_data)


evaluator = BinaryClassificationEvaluator(labelCol='isFraudIndex', metricName='areaUnderROC')
model_score = evaluator.evaluate(predictions)
print(f"Model Score (AUC): {model_score}")


rf_model.save('random_forest_model')


Model Score (AUC): 0.9612481511843898


In [ ]:
rf_model.write().overwrite().save('content/random_forest_model')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>